In [2]:
from spiral import Spiral

sp = Spiral(overrides={
    "cache.enabled": "1",
    "cache.memory_capacity_bytes": "1073741824",  # 1 GiB
    # "cache.disk_capacity_bytes": "10737418240",  # 10 GiB
    "cache.disk_capacity_bytes": "0",
})

In [3]:
project = sp.project("enigma-spiral-poc-555527")

In [5]:
project.list_tables()

[TableResource(id='table_2smfag', project_id='enigma-spiral-poc-555527', dataset='default', table='stim_event_metadata'),
 TableResource(id='table_94rx0u', project_id='enigma-spiral-poc-555527', dataset='default', table='spike_data'),
 TableResource(id='table_9p2h93', project_id='enigma-spiral-poc-555527', dataset='default', table='vidtok_embeddings'),
 TableResource(id='table_bnjrwv', project_id='enigma-spiral-poc-555527', dataset='default', table='stim_constants'),
 TableResource(id='table_hz45n2', project_id='enigma-spiral-poc-555527', dataset='default', table='stim_trial_info'),
 TableResource(id='table_keundd', project_id='enigma-spiral-poc-555527', dataset='default', table='unit_metadata'),
 TableResource(id='table_kpxqb3', project_id='enigma-spiral-poc-555527', dataset='default', table='behavior_adc'),
 TableResource(id='table_ndf1ja', project_id='enigma-spiral-poc-555527', dataset='default', table='probe_metadata'),
 TableResource(id='table_rnzn7k', project_id='enigma-spiral-po

In [6]:
tbl_session_reconstructed_video_metadata = project.table("session_reconstructed_video_metadata")
tbl_spike_data = project.table("spike_data")
tbl_frame_embeddings = project.table("vidtok_embeddings")
tbl_behavior_adc = project.table("behavior_adc")
tbl_stim_event_metadata = project.table("stim_event_metadata")

In [7]:
tbl_spike_data.to_polars_lazy_frame().head().collect()

session_id,probe_id,unit_id,timestamp
str,str,i64,u64


In [8]:
tbl_frame_embeddings.to_polars_lazy_frame().head().collect()

session_id,modality,embedding_index,embedding_tensor,first_frame,first_frame_timestamp,last_frame,last_frame_timestamp,mean_frame_timestamp,source_video_name
str,str,i64,list[list[f32]],i64,u64,i64,u64,u64,str
"""Goliath_2025-10-20_20-40-05""","""depth""",0,"[[-1.093677, -0.114442, … -2.148795], [-1.529309, -0.18276, … -2.316382], … [-0.807859, -0.029813, … -1.955002]]",0,52431216,15,52556341,52493778,"""Goliath_2025-10-20_display_rec…"
"""Goliath_2025-10-20_20-40-05""","""depth""",1,"[[-1.378906, -0.072813, … -2.52733], [-1.717773, -0.178344, … -2.644532], … [-1.118172, 0.231328, … -2.468773]]",16,52564682,31,52689807,52627244,"""Goliath_2025-10-20_display_rec…"
"""Goliath_2025-10-20_20-40-05""","""depth""",2,"[[-1.333009, 0.02928, … -2.716803], [-1.724609, -0.059111, … -2.802735], … [-1.291991, 0.23853, … -2.458994]]",32,52698149,47,52823274,52760711,"""Goliath_2025-10-20_display_rec…"
"""Goliath_2025-10-20_20-40-05""","""depth""",3,"[[-0.320698, -0.004321, … -1.223824], [-0.467016, -0.122193, … -1.262716], … [-1.229397, 0.143292, … -1.642217]]",48,52831616,63,52956839,52894227,"""Goliath_2025-10-20_display_rec…"
"""Goliath_2025-10-20_20-40-05""","""depth""",4,"[[0.449275, -3.974444, … -1.146222], [0.280698, -4.324153, … -1.221712], … [-2.298706, -0.393063, … -2.656256]]",64,52965180,79,53090325,53027752,"""Goliath_2025-10-20_display_rec…"


In [9]:
tbl_behavior_adc.to_polars_lazy_frame().head().collect()

session_id,timestamp,eye_x_px_offset_center,eye_y_px_offset_center,neuropixel_sync_in,photodiode,pupil_size_in,reward_input
str,u64,f64,f64,f64,f64,f64,f64
"""Goliath_2025-10-20_20-40-05""",28317294000000,-753.06663,-282.070811,62062.0,484.0,-96387.0,-329.0
"""Goliath_2025-10-20_20-40-05""",28317460000000,-751.093423,-282.491689,62050.0,545.0,-96381.0,-333.0
"""Goliath_2025-10-20_20-40-05""",28317627000000,-753.105321,-280.193051,62072.0,444.0,-96354.0,-336.0
"""Goliath_2025-10-20_20-40-05""",28317794000000,-752.563656,-281.908935,62068.0,463.0,-96288.0,-340.0
"""Goliath_2025-10-20_20-40-05""",28317960000000,-753.105321,-281.649934,62068.0,479.0,-96379.0,-330.0


In [14]:
import pyarrow as pa

m1 = pa.scalar(1_000_000, type=pa.uint64())

session_ranges = sp.scan({
    "session_id": tbl_session_reconstructed_video_metadata["session_id"],
    "start": tbl_session_reconstructed_video_metadata["timestamp"] * m1,
    "end": (tbl_session_reconstructed_video_metadata["timestamp"] + m1) * m1,
}).to_table()

embeddings_session_ranges = sp.scan({
    "session_id": tbl_session_reconstructed_video_metadata["session_id"],
    "start": tbl_session_reconstructed_video_metadata["timestamp"],
    "end": (tbl_session_reconstructed_video_metadata["timestamp"] + m1),
}).to_table()

In [12]:
snapshot = tbl_behavior_adc.snapshot()

# TODO(marko): Single scan with shards.
def timestamps_behavior_adc_input():
    for r in session_ranges.to_pylist():
        st = pa.scalar(r["start"], type=pa.uint64())
        ed = pa.scalar(r["end"], type=pa.uint64())

        tbl = sp.scan_keys(
            tbl_behavior_adc,
            where=(tbl_behavior_adc["timestamp"] >= st) & (tbl_behavior_adc["timestamp"] < ed) & (tbl_behavior_adc["session_id"] == r["session_id"]),
            asof=snapshot.asof,
        ).to_table()

        array = tbl["timestamp"].to_numpy()
        tss = array[::10]

        yield pa.record_batch({
            "session_id": pa.repeat(r["session_id"], len(tss)),
            "timestamp": tss,
        })

rb = next(timestamps_behavior_adc_input())
rb.schema

session_id: string
timestamp: uint64

In [13]:
# TODO(marko): Remove this when keys are single scan.
ts_batches = []
for i, batch in enumerate(timestamps_behavior_adc_input()):
    if i == 200:
        break
    ts_batches.append(batch)

In [15]:
embeddings_snapshot = tbl_frame_embeddings.snapshot()

def embeddings_input():
    for r in embeddings_session_ranges.to_pylist():
        st = pa.scalar(r["start"], type=pa.uint64())
        ed = pa.scalar(r["end"], type=pa.uint64())

        # TODO(marko): Single scan with shards.
        key_filter = (tbl_frame_embeddings["session_id"] == r["session_id"]) & (tbl_frame_embeddings["modality"] == "rgb")
        tbl = sp.scan_keys(
            tbl_frame_embeddings,
            # TODO(marko): mean_frame_timestamp should be in key
            where=key_filter & (tbl_frame_embeddings["mean_frame_timestamp"] >= st) & (tbl_frame_embeddings["mean_frame_timestamp"] < ed),
            asof=embeddings_snapshot.asof,
        ).to_table()

        yield from tbl.to_reader(max_chunksize=tbl.num_rows)

embeddings_rb = next(embeddings_input())
embeddings_rb.schema

session_id: string
modality: string
embedding_index: int64

In [16]:
# TODO(marko): Remove this when keys are single scan.
ts_embeddings_batches = []
for i, batch in enumerate(embeddings_input()):
    if i == 200:
        break
    ts_embeddings_batches.append(batch)

In [28]:
def ts_infinite_embeddings_batches():
    while True:
        for batch in ts_embeddings_batches:
            yield batch

def ts_infinite_batches():
    while True:
        for batch in ts_batches:
            yield batch

In [32]:
import tqdm

embeddings_scan = sp.scan(tbl_frame_embeddings[["embedding_tensor"]])
behavior_scan = sp.scan(tbl_behavior_adc[["pupil_size_in", "eye_x_px_offset_center", "eye_y_px_offset_center"]])

embeddings_inputs = pa.RecordBatchReader.from_batches(tbl_frame_embeddings.key_schema.to_arrow(), ts_infinite_embeddings_batches())
embeddings_loader = embeddings_scan.to_record_batches(key_table=embeddings_inputs, batch_readahead=32)

behavior_inputs = pa.RecordBatchReader.from_batches(tbl_behavior_adc.key_schema.to_arrow(), ts_infinite_batches())
behavior_loader = behavior_scan.to_record_batches(key_table=behavior_inputs, batch_readahead=32)

for behavior, embeddings in tqdm.tqdm(zip(behavior_loader, embeddings_loader)):
    # TODO(marko): Stack here in numpy.
    # import numpy as np
    # print(f"Batch `behavior` {behavior.num_rows} `embeddings` {embeddings.num_rows}")
    # for tensor in embeddings["embedding_tensor"]:
    #     print(np.array(tensor).shape)
    pass

KeyboardInterrupt: 